# Fake Job Posting Detection

# Model Architecture:

Text → TF-IDF  
        ↓
Categorical → Encoding  
        ↓
Layer 1 → OOF predictions  
        ↓
Layer 2 → Meta models  
        ↓
Layer 3 → Final model

1. Importing the required libraries

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score

2. Loading the dataset

In [ ]:
data = pd.read_csv("/content/CompiledjobListNigeria.csv")
data.fillna('', inplace=True)

3. Different Columns in Dataset

In [ ]:
data.columns

Index(['job_title', 'company_name', 'company_desc', 'job_desc',
       'job_requirement', 'salary', 'location', 'employment_type',
       'department', 'label'],
      dtype='object')

4. Text PreProcessing

In [ ]:
data['text'] = (
    data['job_title'] + " " +
    data['company_desc'] + " " +
    data['job_desc'] + " " +
    data['job_requirement']
)

5. Counting the output class labels

In [ ]:
print(data['label'].value_counts())

label
0    135
1     67
Name: count, dtype: int64


6. TF-IDF ( Term Frequency-Inverse Document Frequency) (TEXT-NUMBERS)

TF-IDF is a mathematical filter that converts text to numbers. It picks out important words that are unique and have high values, while ignoring unimportant words like 'noise' words.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=2000, stop_words='english')

text_features = vectorizer.fit_transform(data['text']).toarray()

7. Encode Categorical

In [ ]:
le = LabelEncoder()

data['location'] = le.fit_transform(data['location'])
data['employment_type'] = le.fit_transform(data['employment_type'])
data['department'] = le.fit_transform(data['department'])

cat_features = data[['location', 'employment_type', 'department']].values

8. Final Features

In [ ]:
X = np.hstack((text_features, cat_features))
y = data['label'].values

9. Train and Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

10. Layer 1 models

In [ ]:
models_layer1 = [
    LogisticRegression(max_iter=1000),
    DecisionTreeClassifier(max_depth=5),
    GaussianNB(),
    SVC(probability=True, kernel='linear')
]

11. OOF Stacking : Is the standard industry "shield" against data leakage.

OOF Stacking prevents data leakage by ensuring the "meta-model" only learns from predictions made on data the base models have never seen.

In [ ]:
from sklearn.model_selection import StratifiedKFold
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

layer1_train = np.zeros((X_train.shape[0], len(models_layer1)))
layer1_test = np.zeros((X_test.shape[0], len(models_layer1)))

for i, model in enumerate(models_layer1):
    test_preds = []

    for train_idx, val_idx in kf.split(X_train, y_train):

        X_tr, X_val = X_train[train_idx], X_train[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]

        model.fit(X_tr, y_tr)

        layer1_train[val_idx, i] = model.predict_proba(X_val)[:, 1]
        test_preds.append(model.predict_proba(X_test)[:, 1])

    layer1_test[:, i] = np.mean(test_preds, axis=0)


12. Layer 2

In [ ]:
model_l2_1 = RandomForestClassifier(n_estimators=100, random_state=42)
model_l2_2 = LogisticRegression(max_iter=1000)

model_l2_1.fit(layer1_train, y_train)
model_l2_2.fit(layer1_train, y_train)

layer2_train = np.column_stack((
    model_l2_1.predict_proba(layer1_train)[:, 1],
    model_l2_2.predict_proba(layer1_train)[:, 1]
))

layer2_test = np.column_stack((
    model_l2_1.predict_proba(layer1_test)[:, 1],
    model_l2_2.predict_proba(layer1_test)[:, 1]
))

13. Final Layer

In [ ]:
final_model = LogisticRegression(max_iter=1000)
final_model.fit(layer2_train, y_train)

y_pred = final_model.predict(layer2_test)

14. Accuracy

In [ ]:
print("Final Accuracy:", accuracy_score(y_test, y_pred))

Final Accuracy: 0.9512195121951219


In [ ]:
print("Actual Classes:   ", y_test)
print("Predicted Classes:", y_pred)

Actual Classes:    [0 1 0 1 0 1 0 1 0 1 1 1 0 0 0 0 1 0 0 0 1 1 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0
 1 0 1 1]
Predicted Classes: [0 1 0 1 0 1 0 1 0 1 1 1 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 1 0 0 0 0 0 0 0 0 1
 1 0 1 1]


14. Printing first 10 predictions

In [ ]:
for i in range(10):  # print first 10
    print("Actual:", y_test[i], "Predicted:", y_pred[i])

Actual: 0 Predicted: 0
Actual: 1 Predicted: 1
Actual: 0 Predicted: 0
Actual: 1 Predicted: 1
Actual: 0 Predicted: 0
Actual: 1 Predicted: 1
Actual: 0 Predicted: 0
Actual: 1 Predicted: 1
Actual: 0 Predicted: 0
Actual: 1 Predicted: 1


In [ ]:
correct = (y_test == y_pred).sum()
wrong = (y_test != y_pred).sum()

print("Correct:", correct)
print("Wrong:", wrong)

Correct: 39
Wrong: 2


15. Overall Classification Report

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.96      0.96      0.96        27
           1       0.93      0.93      0.93        14

    accuracy                           0.95        41
   macro avg       0.95      0.95      0.95        41
weighted avg       0.95      0.95      0.95        41

